In [232]:
%pip install pennylane strawberryfields pennylane-sf scipy==1.10.1




Note: you may need to restart the kernel to use updated packages.


In [233]:
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: c:\Users\Sennai\miniconda3\envs\qrc310\python.exe
Python version: 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]


In [234]:
%pip install pandas scikit-learn openpyxl


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import pandas as pd

# Load
DATA_PATH = Path("QUAC 605 Project Data_processed.xlsx - 3 month_processed.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Place the processed dataset beside this notebook: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("Columns:", df.columns)

# ----------------------------------
# 1. 48-step-ahead forecasting target
# ----------------------------------
df['target'] = df['ENGLAND_WALES_DEMAND'].shift(-48)
df = df.dropna()

# -----------------
# 2. Define features
# ----------------
feature_cols = [
    'SETTLEMENT_PERIOD',
    'EMBEDDED_WIND_GENERATION',
    'EMBEDDED_SOLAR_GENERATION',
    'PUMP_STORAGE_PUMPING',
    'SCOTTISH_TRANSFER',
    'NEMO_FLOW',
    'ELECLINK_FLOW',
    'Temperature',
    'period_sin',
    'period_cos'
]

X_raw = df[feature_cols].values
y_raw = df['target'].values

print("X_raw shape:", X_raw.shape)
print("y_raw shape:", y_raw.shape)


Configuration - QRC


In [ ]:
import numpy as np
import math

import pennylane as qml

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -------------------------
# Cofiguration - optimal for laptop simulation
# ------------------------

SEED_DATA  = 123
SEED_RES   = 42         # base seed for reservoir parameters
N_MODES    = 10          # more modes better to fetaures 
WINDOW     = 48
DEPTH      = 3
CUTOFF_DIM = 4

# Hyperparameters for CV Kerr dynamics
DISP_SCALE = 0.05        # how strongly classical input displaces modes
SQZ_SCALE  = 0.1        # squeezing scale
KERR_SCALE = 0.05       # Kerr strength

rng_data = np.random.default_rng(SEED_DATA)

dev = qml.device(
    "strawberryfields.fock",
    wires=N_MODES,
    cutoff_dim=CUTOFF_DIM,
    shots=None  # analytic expvals
)


print("X_raw shape:", X_raw.shape)
print("y_raw shape:", y_raw.shape)


X_raw shape: (4270, 10)
y_raw shape: (4270,)


Features

In [ ]:
# Time-aware train/val/test split + scaling

n_samples = X_raw.shape[0]
n_train = int(0.7 * n_samples)
n_val   = int(0.85 * n_samples)

# 1) Time-based split (no shuffling)
X_train_raw = X_raw[:n_train]
X_val_raw   = X_raw[n_train:n_val]
X_test_raw  = X_raw[n_val:]

y_train_raw = y_raw[:n_train]
y_val_raw   = y_raw[n_train:n_val]
y_test_raw  = y_raw[n_val:]

print("Train size:", X_train_raw.shape[0])
print("Val size:",   X_val_raw.shape[0])
print("Test size:",  X_test_raw.shape[0])

# 2) Scale X using only train stats
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train_raw)
X_val   = x_scaler.transform(X_val_raw)
X_test  = x_scaler.transform(X_test_raw)

# 3) Scale y as well (for more stable training/metrics)
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train_raw.reshape(-1, 1)).ravel()
y_val   = y_scaler.transform(y_val_raw.reshape(-1, 1)).ravel()
y_test  = y_scaler.transform(y_test_raw.reshape(-1, 1)).ravel()

print("X_train scaled shape:", X_train.shape)
print("y_train scaled shape:", y_train.shape)


Train size: 2989
Val size: 640
Test size: 641
X_train scaled shape: (2989, 10)
y_train scaled shape: (2989,)


Normalization / Random projection

In [ ]:
# Encoder

# Linear encoder from classical features to reservoir modes
n_features = X_train.shape[1]
rng_enc = np.random.default_rng(SEED_RES)
W_enc = rng_enc.normal(loc=0.0, scale=1.0, size=(N_MODES, n_features))  # (6, 9)

print("n_features:", n_features)
print("W_enc shape:", W_enc.shape)

def encode_to_reservoir(X, W_enc):
    """
    X: (num_samples, n_features)
    W_enc: (N_MODES, n_features)
    Returns:
      Z: (num_samples, N_MODES)
    """
    return X @ W_enc.T

Z_train = encode_to_reservoir(X_train, W_enc)
Z_val   = encode_to_reservoir(X_val,   W_enc)
Z_test  = encode_to_reservoir(X_test,  W_enc)

print("Z_train shape:", Z_train.shape)
print("Z_val shape:",   Z_val.shape)
print("Z_test shape:",  Z_test.shape)

def build_windows(Z, y, window):
    """
    Build sliding windows for a single split.

    Z: (T, N_MODES)
    y: (T,)
    window: int

    Returns:
      Z_windows: (T - window, window, N_MODES)
      y_targets: (T - window,)
    """
    T = Z.shape[0]
    num_samples = T - window
    Z_w = np.zeros((num_samples, window, Z.shape[1]))
    y_t = np.zeros(num_samples)

    for i in range(num_samples):
        Z_w[i] = Z[i : i + window]        # [t ... t+window-1]
        y_t[i] = y[i + window]            # target at t+window

    return Z_w, y_t

Z_windows_train, y_targets_train = build_windows(Z_train, y_train, WINDOW)
Z_windows_val,   y_targets_val   = build_windows(Z_val,   y_val,   WINDOW)
Z_windows_test,  y_targets_test  = build_windows(Z_test,  y_test,  WINDOW)

print("Z_windows_train:", Z_windows_train.shape)
print("y_targets_train:", y_targets_train.shape)
print("Z_windows_val:",   Z_windows_val.shape)
print("y_targets_val:",   y_targets_val.shape)
print("Z_windows_test:",  Z_windows_test.shape)
print("y_targets_test:",  y_targets_test.shape)



n_features: 10
W_enc shape: (10, 10)
Z_train shape: (2989, 10)
Z_val shape: (640, 10)
Z_test shape: (641, 10)
Z_windows_train: (2941, 48, 10)
y_targets_train: (2941,)
Z_windows_val: (592, 48, 10)
y_targets_val: (592,)
Z_windows_test: (593, 48, 10)
y_targets_test: (593,)


Classical (RF) baseline for QRC Configuration Optimazation based on CV encoding

In [ ]:
#Classical RF baseline on flattened windows (same split/horizon as QRC)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import math

# 1) Build classical windows directly from scaled X (no QRC)
Xw_train, y_cls_train = build_windows(X_train, y_train, WINDOW)
Xw_val,   y_cls_val   = build_windows(X_val,   y_val,   WINDOW)
Xw_test,  y_cls_test  = build_windows(X_test,  y_test,  WINDOW)

print("Xw_train shape:", Xw_train.shape)  # (T_train - WINDOW, WINDOW, n_features)

# 2) Flatten window dimension: (num_samples, WINDOW * n_features)
n_train_w = Xw_train.shape[0]
n_val_w   = Xw_val.shape[0]
n_test_w  = Xw_test.shape[0]

Xw_train_flat = Xw_train.reshape(n_train_w, -1)
Xw_val_flat   = Xw_val.reshape(n_val_w,   -1)
Xw_test_flat  = Xw_test.reshape(n_test_w, -1)

print("Xw_train_flat shape:", Xw_train_flat.shape)

# 3) Train Random Forest on flattened windows
rf_base = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rf_base.fit(Xw_train_flat, y_cls_train)

# 4) Predictions
y_train_pred_base = rf_base.predict(Xw_train_flat)
y_val_pred_base   = rf_base.predict(Xw_val_flat)
y_test_pred_base  = rf_base.predict(Xw_test_flat)

# 5) R² scores
r2_train_base = r2_score(y_cls_train, y_train_pred_base)
r2_val_base   = r2_score(y_cls_val,   y_val_pred_base)
r2_test_base  = r2_score(y_cls_test,  y_test_pred_base)

# 6) Error metrics (on scaled y)
mae_val_base  = mean_absolute_error(y_cls_val,  y_val_pred_base)
rmse_val_base = math.sqrt(mean_squared_error(y_cls_val,  y_val_pred_base))

mae_test_base  = mean_absolute_error(y_cls_test,  y_test_pred_base)
rmse_test_base = math.sqrt(mean_squared_error(y_cls_test, y_test_pred_base))

# 7) Output
print(f"[CLASSICAL RF] TRAIN R²: {r2_train_base:.3f}")
print(f"[CLASSICAL RF] VAL   R²: {r2_val_base:.3f}, MAE: {mae_val_base:.3f}, RMSE: {rmse_val_base:.3f}")
print(f"[CLASSICAL RF] TEST  R²: {r2_test_base:.3f}, MAE: {mae_test_base:.3f}, RMSE: {rmse_test_base:.3f}")


Xw_train shape: (2941, 48, 10)
Xw_train_flat shape: (2941, 480)
[CLASSICAL RF] TRAIN R²: 0.984
[CLASSICAL RF] VAL   R²: 0.608, MAE: 0.351, RMSE: 0.516
[CLASSICAL RF] TEST  R²: 0.054, MAE: 0.516, RMSE: 0.713


Kerr CV-QRC

In [ ]:
# ==============================================
# KERR CV-QRC: SIMPLE, SINGLE-SNAPSHOT RESERVOIR
# ==============================================

# 1) Parameter init
phi = None
theta_bs = None
phi_bs = None
r_sqz = None
kappa = None

def init_reservoir_params(seed):
    global phi, theta_bs, phi_bs, r_sqz, kappa

    rng_res = np.random.default_rng(seed)

    # Rotations
    phi = rng_res.uniform(
        low=0.0, high=2*np.pi, size=(DEPTH, N_MODES)
    )

    # Beamsplitters
    theta_bs = rng_res.normal(
        loc=0.0, scale=np.pi/10, size=(DEPTH, N_MODES-1)
    )
    phi_bs = rng_res.uniform(
        low=0.0, high=2*np.pi, size=(DEPTH, N_MODES-1)
    )

    # Squeezing
    r_sqz = rng_res.normal(
        loc=0.0, scale=SQZ_SCALE, size=(DEPTH, N_MODES)
    )

    # Kerr
    kappa = rng_res.normal(
        loc=0.0, scale=KERR_SCALE / np.sqrt(DEPTH), size=(DEPTH, N_MODES)
    )

# Initialize once
init_reservoir_params(SEED_RES)

@qml.qnode(dev, diff_method=None)
def kerr_qrc_window(z_flat):
    """
    CV-QRC for an entire window.
    We evolve over WINDOW steps, then measure X on all modes ONCE at the end.
    """
    z_flat = np.array(z_flat, dtype=float)
    z_window = z_flat.reshape(WINDOW, N_MODES)

    for t in range(WINDOW):
        z_t = z_window[t]

        # Input encoding
        for i in range(N_MODES):
            qml.Displacement(DISP_SCALE * z_t[i], 0.0, wires=i)

        # Reservoir dynamics
        for d in range(DEPTH):

            # Rotations
            for i in range(N_MODES):
                qml.Rotation(phi[d, i], wires=i)

            # Beamsplitters
            for i in range(N_MODES - 1):
                qml.Beamsplitter(theta_bs[d, i], phi_bs[d, i], wires=[i, i+1])

            # Squeezing
            for i in range(N_MODES):
                qml.Squeezing(r_sqz[d, i], 0.0, wires=i)

            # Kerr
            for i in range(N_MODES):
                qml.Kerr(kappa[d, i], wires=i)

    return [qml.expval(qml.X(i)) for i in range(N_MODES)]


def compute_hidden_states_from_windows_kerr(Z_windows):
    """
    Z_windows: (num_samples, WINDOW, N_MODES)
    Returns:
      H: (num_samples, N_MODES)
    """
    num_samples = Z_windows.shape[0]
    H = np.zeros((num_samples, N_MODES))

    for i in range(num_samples):
        z_win_flat = Z_windows[i].reshape(-1)
        H[i, :] = kerr_qrc_window(z_win_flat)

        if (i + 1) % 10 == 0:
            print(f"[Kerr] window {i+1}/{num_samples}", end="\r")
    print()
    return H

print("\n=== Computing H_kerr (simple snapshot) ===")
H_train = compute_hidden_states_from_windows_kerr(Z_windows_train)
H_val   = compute_hidden_states_from_windows_kerr(Z_windows_val)
H_test  = compute_hidden_states_from_windows_kerr(Z_windows_test)

print("H_train shape:", H_train.shape)  # (T_train - WINDOW, N_MODES)
print("H_val shape:",   H_val.shape)
print("H_test shape:",  H_test.shape)



=== Computing H_kerr (simple snapshot) ===


MemoryError: Unable to allocate 16.0 TiB for an array with shape (4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4) and data type complex128

Compute hidden state

In [ ]:
#Compute hidden states + ensemble wrapper

def compute_hidden_states_from_windows_kerr(Z_windows):
    """
    Z_windows: (num_samples, WINDOW, N_MODES)
    Returns:
      H: (num_samples, N_MODES)
    """
    num_samples = Z_windows.shape[0]
    H = np.zeros((num_samples, N_MODES))

    for i in range(num_samples):
        z_win_flat = Z_windows[i].reshape(-1)  # length = WINDOW * N_MODES
        H[i, :] = kerr_qrc_window(z_win_flat)  # vector of size N_MODES

        if (i + 1) % 10 == 0:
            print(f"[Kerr] computed window {i+1}/{num_samples}", end="\r")
    print()
    return H

def compute_H_ensemble(Z_windows, n_ensembles=1):
    """
    Build an ensemble of reservoirs by re-initializing parameters with different seeds
    and concatenating the resulting hidden states.

    Z_windows: (num_samples, WINDOW, N_MODES)
    n_ensembles: int

    Returns:
      H_concat: (num_samples, N_MODES * n_ensembles)
    """
    H_list = []
    for ens in range(n_ensembles):
        print(f"\n=== Ensemble {ens+1}/{n_ensembles} ===")
        # re-init reservoir params with a new seed
        init_reservoir_params(SEED_RES + ens)
        H_ens = compute_hidden_states_from_windows_kerr(Z_windows)
        H_list.append(H_ens)

    H_concat = np.concatenate(H_list, axis=1)
    print("Ensemble H shape:", H_concat.shape)
    return H_concat

N_ENSEMBLES = 1  # more for richer features

H_train = compute_H_ensemble(Z_windows_train, n_ensembles=N_ENSEMBLES)
H_val   = compute_H_ensemble(Z_windows_val,   n_ensembles=N_ENSEMBLES)
H_test  = compute_H_ensemble(Z_windows_test,  n_ensembles=N_ENSEMBLES)

print("H_train shape:", H_train.shape)
print("H_val shape:",   H_val.shape)
print("H_test shape:",  H_test.shape)


In [ ]:
#Scaling of H for linear models

H_scaler = StandardScaler()
H_train_scaled = H_scaler.fit_transform(H_train)
H_val_scaled   = H_scaler.transform(H_val)
H_test_scaled  = H_scaler.transform(H_test)

print("H_train_scaled shape:", H_train_scaled.shape)
